## Applied Machine Learning Homework 2
### Shane Wang tsw2134

In [1]:
# !pip install scikit-learn pandas numpy matplotlib seaborn xgboost

### 1. Data Preparation

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import(
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import learning_curve, validation_curve

We will be using the Home Credit Default Risk Dataset, found on https://www.kaggle.com/c/home-credit-default-risk, for this assignment.

In [3]:
df = pd.read_csv('home-credit-default-risk/application_train.csv')
df.shape

(307511, 122)

In [4]:
df.head(15)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.000,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.000,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.000,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.000,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.000,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
5,100008,0,Cash loans,M,N,Y,0,99000.000,490495.5,27517.5,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
6,100009,0,Cash loans,F,Y,Y,1,171000.000,1560726.0,41301.0,...,0,0,0,0,0.0,0.0,0.0,1.0,1.0,2.0
7,100010,0,Cash loans,M,Y,Y,0,360000.000,1530000.0,42075.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
8,100011,0,Cash loans,F,N,Y,0,112500.000,1019610.0,33826.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
9,100012,0,Revolving loans,M,N,Y,0,135000.000,405000.0,20250.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


#### Splitting Train, Validation, and Test Sets

In [5]:
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

# First split: 70% Train, 30% Temp (which will be Val + Test)
# df_train, df_temp = train_test_split(df, test_size=0.3, random_state=42)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

# Second split: Split the 30% Temp into two equal 15% halves
# df_validation, df_test = train_test_split(df_temp, test_size=0.5, random_state=42)
X_validation, X_test, y_validation, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [6]:
# df_train["TARGET"].value_counts(normalize=True) # see if the train, validation, and test sets are approx. equally split for randomness
y_train.value_counts(normalize=True)

TARGET
0    0.919111
1    0.080889
Name: proportion, dtype: float64

In [7]:
# df_validation["TARGET"].value_counts(normalize=True)
y_validation.value_counts(normalize=True)

TARGET
0    0.920242
1    0.079758
Name: proportion, dtype: float64

In [8]:
# df_test["TARGET"].value_counts(normalize=True)
y_test.value_counts(normalize=True)

TARGET
0    0.91905
1    0.08095
Name: proportion, dtype: float64

#### Dropping Columns with A Lot of Nulls

In [9]:
# Find numerical and categorical columns from training set; Claude helped with this, as this wasn't intuitive
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features:     {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")
print(f"\nCategorical columns: {cat_cols}")

Numeric features:     104
Categorical features: 16

Categorical columns: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


/var/folders/5z/mlflw5dd4k71m06j1vl7bhxh0000gn/T/ipykernel_50445/1521556572.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()


In [10]:
# Inspect missing values in the training set only
train_missing = X_train.isnull().sum()
train_missing_pct = (train_missing / len(X_train) * 100).round(2)
missing_summary = pd.DataFrame({
    'Missing Count': train_missing,
    'Missing %': train_missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print(f"Features with missing values: {len(missing_summary)}")
print(missing_summary.head(30).to_string())

Features with missing values: 67
                          Missing Count  Missing %
COMMONAREA_MEDI                  150377      69.86
COMMONAREA_AVG                   150377      69.86
COMMONAREA_MODE                  150377      69.86
NONLIVINGAPARTMENTS_MEDI         149407      69.41
NONLIVINGAPARTMENTS_MODE         149407      69.41
NONLIVINGAPARTMENTS_AVG          149407      69.41
FONDKAPREMONT_MODE               147183      68.38
LIVINGAPARTMENTS_MODE            147079      68.33
LIVINGAPARTMENTS_MEDI            147079      68.33
LIVINGAPARTMENTS_AVG             147079      68.33
FLOORSMIN_MODE                   145996      67.82
FLOORSMIN_MEDI                   145996      67.82
FLOORSMIN_AVG                    145996      67.82
YEARS_BUILD_MODE                 143139      66.50
YEARS_BUILD_MEDI                 143139      66.50
YEARS_BUILD_AVG                  143139      66.50
OWN_CAR_AGE                      141836      65.89
LANDAREA_AVG                     127807      59.3

In [11]:
# Drop columns with >60% missing in the training set (too sparse to impute reliably)
high_missing_thresh = 60.0
high_missing_cols = missing_summary[missing_summary['Missing %'] > high_missing_thresh].index.tolist()
print(f"Dropping {len(high_missing_cols)} columns with >{high_missing_thresh}% missing:")
print(high_missing_cols)

X_train = X_train.drop(columns=high_missing_cols)
X_validation = X_validation.drop(columns=high_missing_cols)
X_test  = X_test.drop(columns=high_missing_cols)

# Refresh column lists after dropping
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
print(f"\nRemaining features: {X_train.shape[1]}  (numeric: {len(num_cols)}, categorical: {len(cat_cols)})")

Dropping 17 columns with >60.0% missing:
['COMMONAREA_MEDI', 'COMMONAREA_AVG', 'COMMONAREA_MODE', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAPARTMENTS_AVG', 'FONDKAPREMONT_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAPARTMENTS_AVG', 'FLOORSMIN_MODE', 'FLOORSMIN_MEDI', 'FLOORSMIN_AVG', 'YEARS_BUILD_MODE', 'YEARS_BUILD_MEDI', 'YEARS_BUILD_AVG', 'OWN_CAR_AGE']

Remaining features: 103  (numeric: 88, categorical: 15)


/var/folders/5z/mlflw5dd4k71m06j1vl7bhxh0000gn/T/ipykernel_50445/3712796539.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()


In [12]:
# Refresh the missing count again (after dropping)
train_missing = X_train.isnull().sum()
train_missing_pct = (train_missing / len(X_train) * 100).round(2)
missing_summary = pd.DataFrame({
    'Missing Count': train_missing,
    'Missing %': train_missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print(f"Features with missing values: {len(missing_summary)}")
print(missing_summary.head(30).to_string())

Features with missing values: 50
                              Missing Count  Missing %
LANDAREA_AVG                         127807      59.37
LANDAREA_MEDI                        127807      59.37
LANDAREA_MODE                        127807      59.37
BASEMENTAREA_MODE                    125984      58.53
BASEMENTAREA_AVG                     125984      58.53
BASEMENTAREA_MEDI                    125984      58.53
EXT_SOURCE_1                         121249      56.33
NONLIVINGAREA_MODE                   118888      55.23
NONLIVINGAREA_AVG                    118888      55.23
NONLIVINGAREA_MEDI                   118888      55.23
ELEVATORS_MEDI                       114721      53.29
ELEVATORS_AVG                        114721      53.29
ELEVATORS_MODE                       114721      53.29
WALLSMATERIAL_MODE                   109480      50.86
APARTMENTS_MEDI                      109300      50.78
APARTMENTS_MODE                      109300      50.78
APARTMENTS_AVG                  

#### Fixing Inconsistent Data Types

In [13]:
df[['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION']].head(10)

,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION
0,-9461,-637,-3648.0
1,-16765,-1188,-1186.0
2,-19046,-225,-4260.0
3,-19005,-3039,-9833.0
4,-19932,-3038,-4311.0
5,-16941,-1588,-4970.0
6,-13778,-3130,-1213.0
7,-18850,-449,-4597.0
8,-20099,365243,-7427.0
9,-14469,-2019,-14437.0


In [14]:
# DAYS_* columns are stored as negative integers (days before application).
# Convert to positive absolute values so they read naturally as durations.
# Claude helped me here, since it would've taken me a long time on StackOverflow or something to find out how to filter all columns that contain 'DAYS' in the string name
days_cols = [c for c in num_cols if 'DAYS' in c]
print(len(days_cols))

for col in days_cols:
    X_train[col] = X_train[col].abs()
    X_validation[col] = X_validation[col].abs()
    X_test[col] = X_test[col].abs()

# DAYS_EMPLOYED = 365243 means "unemployed", so replace with NaN so it doesn't inflate employment duration to 1000 years.
if 'DAYS_EMPLOYED' in num_cols:
    sentinel = 365243
    for split in [X_train, X_validation, X_test]:
        split['DAYS_EMPLOYED'] = split['DAYS_EMPLOYED'].replace(sentinel, np.nan)

5


#### Feature Engineering

In [15]:
# Claude helped give me suggestions for which features are necessary
def add_features(df):
    df = df.copy()
    # Credit-to-income burden: how many years of income does the loan represent?
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1)
    # Annual payment burden as share of income
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
    # Implicit loan term in years (credit / annuity)
    df['CREDIT_TERM'] = df['AMT_CREDIT'] / (df['AMT_ANNUITY'] + 1)
    # Goods price relative to loan amount
    df['GOODS_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / (df['AMT_CREDIT'] + 1)
    # Age in years
    df['AGE_YEARS'] = df['DAYS_BIRTH'] / 365.0
    # Employment tenure in years
    df['YEARS_EMPLOYED'] = df['DAYS_EMPLOYED'] / 365.0
    # Age group buckets
    df['AGE_GROUP'] = pd.cut(
        df['AGE_YEARS'],
        bins=[0, 25, 35, 45, 55, 100],
        labels=[0, 1, 2, 3, 4]  # encoded as int to avoid object dtype
    ).astype(float)
    return df

In [16]:
X_train = add_features(X_train)
X_validation = add_features(X_validation)
X_test = add_features(X_test)

# Refresh numeric and categorical columns
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features:     {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")
print(f"\nCategorical columns: {cat_cols}")

Numeric features:     95
Categorical features: 15

Categorical columns: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


/var/folders/5z/mlflw5dd4k71m06j1vl7bhxh0000gn/T/ipykernel_50445/4281486254.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()


#### Missing Value Imputation

In [17]:
from sklearn.impute import SimpleImputer
# Claude taught me how to use sklearn's SimpleImputer

# I am fitting only on training data, then transform validation and test sets with same statistics to avoid data leakage

# Numeric: median imputation (great for incomes, since it's exponential for long tail) 
# - I am NOT doing group-wise medians because that would be way too complicated
num_imputer = SimpleImputer(strategy='median')
X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols]) # this method does a fit AND then a transform, i.e. fit(...).transform(...)
X_validation[num_cols] = num_imputer.transform(X_validation[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

# Categorical: most-frequent (mode) imputation
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_validation[cat_cols] = cat_imputer.transform(X_validation[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

# Test cases
assert X_train.isnull().sum().sum() == 0
assert X_validation.isnull().sum().sum() == 0
assert X_test.isnull().sum().sum() == 0

#### Categorical Encoding

In [22]:
from sklearn.preprocessing import LabelEncoder
# Fit LabelEncoder on training categories only. Unseen categories in val/test are mapped to -1 via the fallback below.

label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    le.fit(X_train[col].astype(str)) # train by fitting on training set, then add transform to the training, validation, and test
    label_encoders[col] = le

    X_train[col] = le.transform(X_train[col].astype(str))

    for split in [X_validation, X_test]:
        split[col] = split[col].astype(str).map(
            lambda x, le=le: le.transform([x])[0]
            if x in le.classes_ else -1 # if cat DNE in training, then label as -1
        )

#### Feature Scaling

In [26]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_validation_scaled = scaler.transform(X_validation)
X_test_scaled = scaler.transform(X_test)

# Keep as DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_validation_scaled = pd.DataFrame(X_validation_scaled, columns=X_validation.columns, index=X_validation.index)
X_test_scaled  = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

#### Data Prep Summary

In [32]:
# Claude was used to generate the following boilerplate information, so I didn't have to spend time fixing formatting:
print(f"  Train rows:      {len(X_train):>7,}  |  Target mean: {y_train.mean()*100:.2f}%")
print(f"  Validation rows: {len(X_validation):>7,}  |  Target mean: {y_validation.mean()*100:.2f}%")
print(f"  Test rows:       {len(X_test):>7,}  |  Target mean: {y_test.mean()*100:.2f}%")
print(f"  Features:        {X_train.shape[1]:>7,}")
print()
print("  Available sets:")
print("    X_train / y_train           for GBDT (unscaled)")
print("    X_val   / y_val")
print("    X_test  / y_test")
print("    X_train_scaled / y_train    for MLP (scaled)")
print("    X_val_scaled   / y_val")
print("    X_test_scaled  / y_test")

  Train rows:      215,257  |  Target mean: 8.09%
  Validation rows:  46,127  |  Target mean: 7.98%
  Test rows:        46,127  |  Target mean: 8.10%
  Features:            110

  Available sets:
    X_train / y_train           for GBDT (unscaled)
    X_val   / y_val
    X_test  / y_test
    X_train_scaled / y_train    for MLP (scaled)
    X_val_scaled   / y_val
    X_test_scaled  / y_test


### 2. Gradient Boosted Tree (GBDT)

In [ ]:
import xgboost as xgb

### 3. Multi-Layer Perceptron (MLP)

### 4. GBDT vs MLP Comparison